# Notebook 10 — Feature engineering 🛠️

En el **NB09** convertiste variables categóricas en columnas numéricas y mejoraste el modelo. Hoy vas a hacer algo todavía más poderoso: **inventar features nuevas** que no estaban en los datos originales pero que **revelan información oculta**.

Esto se llama **feature engineering** y suele aportar más mejora al modelo que cambiar de algoritmo. Es uno de los superpoderes del data scientist.

## Idea central

```
    sibsp + parch + 1   →   family_size   (¿cuántos viajan juntos?)
    "Mr. John Smith"    →   title="Mr"    (señal social oculta en el nombre)
    age = 8             →   age_group="niño"
    fare = 250          →   log_fare ≈ 5.52   (suaviza valores extremos)
```

## Objetivos de aprendizaje

1. Cargar una versión del Titanic con la columna **`name`** (la versión de seaborn no la trae).
2. Combinar columnas existentes para crear features más informativas: **`family_size`**, **`is_alone`**, **`fare_per_person`**.
3. **Discretizar** una variable continua en bins con `pd.cut`: edad → grupo de edad.
4. **Extraer información de strings** con `.str.extract()` + regex: nombre → título social.
5. **Transformar logarítmicamente** una variable muy asimétrica con `np.log1p`.
6. Entrenar un modelo con las nuevas features y comparar el R².

---

## 1. Setup — cargar el Titanic con `name`

La versión de `seaborn` no incluye el nombre del pasajero. Cargamos una versión pública desde una URL estable que sí lo trae.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)

# Normalize column names to match the conventions of the previous notebooks
df.columns = df.columns.str.lower()

print(f"df: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

### Limpieza ligera

Imputamos la mediana en `age` y descartamos las pocas filas con `embarked` faltante.

In [ ]:
df_clean = df.copy()
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean = df_clean.dropna(subset=["embarked"]).reset_index(drop=True)

print(f"df_clean: {df_clean.shape[0]} rows")
df_clean[["name", "sex", "age", "sibsp", "parch", "fare", "embarked"]].head(3)

---

## 2. Feature combinada — `family_size` y `is_alone`

`sibsp` (hermanos/cónyuge) y `parch` (padres/hijos) por separado son menos informativos que **el tamaño total del grupo familiar a bordo**:

```
    family_size = sibsp + parch + 1     # +1 = el propio pasajero
```

Y a partir de eso, una feature binaria que captura algo distinto: viajar solo es muy diferente a viajar acompañado.

```
    is_alone = (family_size == 1)
```

### 🏋️ Ejercicio 1 — Crear `family_size` y `is_alone`

1. Añade a `df_clean` una columna **`family_size`** = `sibsp + parch + 1`.
2. Añade una columna **`is_alone`** que valga `1` cuando `family_size == 1` y `0` en cualquier otro caso (tipo `int`).

In [ ]:
# YOUR CODE HERE
df_clean["family_size"] = None
df_clean["is_alone"] = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert "family_size" in df_clean.columns, "Column 'family_size' is missing"
assert "is_alone" in df_clean.columns, "Column 'is_alone' is missing"

# family_size must be at least 1 (the passenger themselves)
assert df_clean["family_size"].min() >= 1, "family_size must always be >= 1 (the passenger counts!)"
assert df_clean["family_size"].max() <= 20, f"family_size max ({df_clean['family_size'].max()}) seems too large"

# Manual sanity check: family_size must equal sibsp + parch + 1
expected = df_clean["sibsp"] + df_clean["parch"] + 1
assert (df_clean["family_size"] == expected).all(), "family_size != sibsp + parch + 1"

# is_alone must be 0/1 integer
assert set(df_clean["is_alone"].unique().tolist()) <= {0, 1}, "is_alone must contain only 0/1"
assert df_clean["is_alone"].dtype.kind in "iu", f"is_alone must be integer dtype, got {df_clean['is_alone'].dtype}"

# is_alone == 1 iff family_size == 1
mask_alone = df_clean["family_size"] == 1
assert (df_clean.loc[mask_alone, "is_alone"] == 1).all(), "is_alone should be 1 when family_size==1"
assert (df_clean.loc[~mask_alone, "is_alone"] == 0).all(), "is_alone should be 0 when family_size>1"

print("✅ ¡Todos los tests pasaron! Tienes 2 features nuevas a partir de columnas viejas.")
print(f"   Pasajeros que viajaban solos: {df_clean['is_alone'].sum()} de {len(df_clean)} "
      f"({df_clean['is_alone'].mean()*100:.1f}%)")

---

## 3. Feature derivada — `fare_per_person`

El `fare` que figura en los datos es el **precio total del billete**, no por persona. Para una familia de 4 que paga 100, el coste por persona es 25 — radicalmente distinto a un pasajero solo que paga 100.

```
    fare_per_person = fare / family_size
```

### 🏋️ Ejercicio 2 — Crear `fare_per_person`

Añade a `df_clean` una columna **`fare_per_person`** = `fare / family_size`.

In [ ]:
# YOUR CODE HERE
df_clean["fare_per_person"] = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert "fare_per_person" in df_clean.columns, "Column 'fare_per_person' is missing"

# Numeric dtype
assert df_clean["fare_per_person"].dtype.kind == "f", \
    f"fare_per_person must be float, got {df_clean['fare_per_person'].dtype}"

# Manual check: fare_per_person must equal fare / family_size
expected = df_clean["fare"] / df_clean["family_size"]
assert np.allclose(df_clean["fare_per_person"], expected, equal_nan=True), \
    "fare_per_person != fare / family_size"

# Should be ≤ fare for every passenger
assert (df_clean["fare_per_person"] <= df_clean["fare"] + 1e-9).all(), \
    "fare_per_person must be <= fare for every row"

# Sanity: must be ≥ 0
assert (df_clean["fare_per_person"].fillna(0) >= 0).all(), "fare_per_person must be >= 0"

print("✅ ¡Tests pasaron! Tienes el precio del billete normalizado por pasajero.")
print(f"   fare media:             {df_clean['fare'].mean():.2f}")
print(f"   fare_per_person media:  {df_clean['fare_per_person'].mean():.2f}")

---

## 4. Discretización — convertir `age` en `age_group` con `pd.cut`

A veces no necesitas la edad exacta, sino una **categoría de edad**. Esto se llama *binning* o *discretización* y se hace con `pd.cut`.

```python
pd.cut(
    df_clean["age"],
    bins=[0, 12, 18, 60, 100],
    labels=["niño", "adolescente", "adulto", "mayor"],
)
```

- **`bins`**: bordes de los intervalos. `[0, 12, 18, 60, 100]` define 4 intervalos: (0, 12], (12, 18], (18, 60], (60, 100].
- **`labels`**: nombre de cada intervalo.

> 💡 ¿Cuándo conviene discretizar? Cuando la relación con el target **no es lineal** — por ejemplo, si los niños tienen mayor probabilidad de sobrevivir que los adolescentes, pero los adolescentes y adultos jóvenes son similares, agruparlos puede ayudar al modelo.

### 🏋️ Ejercicio 3 — Crear `age_group`

Añade a `df_clean` una columna **`age_group`** con `pd.cut` usando los bins `[0, 12, 18, 60, 100]` y los labels `["niño", "adolescente", "adulto", "mayor"]`.

In [ ]:
# YOUR CODE HERE
df_clean["age_group"] = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert "age_group" in df_clean.columns, "Column 'age_group' is missing"

# Should be a categorical column
expected_labels = {"niño", "adolescente", "adulto", "mayor"}
actual_labels = set(df_clean["age_group"].dropna().unique().tolist())
assert actual_labels <= expected_labels, \
    f"age_group labels must be a subset of {expected_labels}, got {actual_labels}"
assert "niño" in actual_labels, "Expected to see at least some 'niño' rows"
assert "adulto" in actual_labels, "Expected to see at least some 'adulto' rows"

# Manual cross-check: a 5-year-old should be 'niño'
child_rows = df_clean[df_clean["age"] <= 12]
if len(child_rows) > 0:
    assert (child_rows["age_group"] == "niño").all(), \
        "All passengers aged <= 12 must be in the 'niño' bin"

# Manual cross-check: a 70-year-old should be 'mayor'
senior_rows = df_clean[df_clean["age"] > 60]
if len(senior_rows) > 0:
    assert (senior_rows["age_group"] == "mayor").all(), \
        "All passengers aged > 60 must be in the 'mayor' bin"

print("✅ ¡Tests pasaron! Edad agrupada en bins.")
print(df_clean["age_group"].value_counts())

---

## 5. Extraer información de strings — `title` desde `name`

Los nombres del Titanic siguen un formato bastante fijo:

```
    "Braund, Mr. Owen Harris"
    "Cumings, Mrs. John Bradley (Florence Briggs Thayer)"
    "Heikkinen, Miss. Laina"
```

El **título social** (`Mr`, `Mrs`, `Miss`, `Master`, etc.) está justo después de la primera coma y termina con un punto. Es una señal súper informativa: `Master` indica niño varón, `Mrs` mujer casada, etc.

### Cómo extraerlo con regex

```python
df_clean["title"] = df_clean["name"].str.extract(r", ([A-Za-z]+)\.")
```

| Pieza del regex | Significado |
|---|---|
| `, ` | Una coma y un espacio (después del apellido) |
| `(...)` | El **grupo de captura** — lo que `.str.extract` devuelve |
| `[A-Za-z]+` | Una o más letras (el título en sí) |
| `\.` | Un punto literal (escapamos el `.` porque en regex significa "cualquier carácter") |

> 💡 Si nunca has visto regex, no te preocupes: en el bootcamp lo verás a fondo. Por ahora basta con saber que es una **mini-lengua** para describir patrones en texto.

### ⚠️ Casos raros

Algunos nombres tienen títulos compuestos (`"the Countess"`) que no encajan en el regex simple. Después del `.str.extract` rellenamos esos casos con `"Other"` usando `.fillna(...)`:

```python
df_clean["title"] = df_clean["title"].fillna("Other")
```

Esta es una situación muy típica del trabajo con datos reales: tu primer regex captura el 99% de los casos y dejas un cubo `"Other"` para el resto.

### 🏋️ Ejercicio 4 — Extraer el título

1. Crea una columna **`title`** en `df_clean` aplicando `.str.extract(r", ([A-Za-z]+)\.")` sobre `name`.
2. Aplica `.fillna("Other")` sobre la columna para cubrir los casos raros.

In [ ]:
# YOUR CODE HERE
df_clean["title"] = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert "title" in df_clean.columns, "Column 'title' is missing"

# After fillna, there should be no NaN
assert df_clean["title"].isna().sum() == 0, \
    f"Found {df_clean['title'].isna().sum()} NaN — did you forget the .fillna('Other')?"

# Should be a Series of strings (object or StringDtype both work)
assert pd.api.types.is_string_dtype(df_clean["title"]), \
    f"title must be string-like dtype, got {df_clean['title'].dtype}"

# The 4 most common titles in the Titanic dataset
common = {"Mr", "Miss", "Mrs", "Master"}
extracted_top4 = set(df_clean["title"].value_counts().head(4).index.tolist())
assert extracted_top4 == common, \
    f"Top 4 titles should be {common}, got {extracted_top4}"

# At least one row should be 'Other' (the Countess)
assert (df_clean["title"] == "Other").sum() >= 1, \
    "Expected at least 1 row with title == 'Other' (e.g. 'the Countess')"

# Sanity: 'Mr' should be the most common title (~500+ passengers)
assert (df_clean["title"] == "Mr").sum() > 400, \
    f"Expected >400 'Mr', got {(df_clean['title']=='Mr').sum()}"

print("✅ ¡Tests pasaron! Título social extraído del nombre.")
print(df_clean["title"].value_counts())

---

## 6. Transformación logarítmica de `fare`

Mira la distribución de `fare`. Es **muy asimétrica**: la mayoría paga poco, pero hay unos pocos billetes carísimos que estiran la cola hacia la derecha. Esa asimetría confunde a la regresión lineal.

La **transformación logarítmica** aplasta los valores grandes y deja la distribución más simétrica.

```python
log_fare = np.log1p(fare)     # log(1 + fare) — log1p evita problemas con fare = 0
```

### 🏋️ Ejercicio 5 — Crear `log_fare`

Añade una columna **`log_fare`** = `np.log1p(df_clean["fare"])`.

In [ ]:
# YOUR CODE HERE
df_clean["log_fare"] = None

In [ ]:
# Tests — verify the exercise was completed correctly
assert "log_fare" in df_clean.columns, "Column 'log_fare' is missing"
assert df_clean["log_fare"].dtype.kind == "f", \
    f"log_fare must be float, got {df_clean['log_fare'].dtype}"

# log_fare must equal log1p(fare)
expected = np.log1p(df_clean["fare"])
assert np.allclose(df_clean["log_fare"], expected, equal_nan=True), \
    "log_fare != np.log1p(fare)"

# Skew should drop significantly
skew_raw = df_clean["fare"].skew()
skew_log = df_clean["log_fare"].skew()
assert abs(skew_log) < abs(skew_raw), \
    f"log_fare should be less skewed than fare (|skew_log|={abs(skew_log):.2f} vs |skew_raw|={abs(skew_raw):.2f})"

print(f"✅ ¡Tests pasaron! Skew de fare:     {skew_raw:.2f}")
print(f"                  Skew de log_fare: {skew_log:.2f}  ← mucho más simétrica")

### Visualización del antes/después

Comparamos los histogramas de `fare` y `log_fare`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_clean["fare"], bins=40, edgecolor="black")
axes[0].set_title(f"fare (skew = {df_clean['fare'].skew():.2f})")
axes[0].set_xlabel("fare")
axes[0].set_ylabel("frecuencia")

axes[1].hist(df_clean["log_fare"], bins=40, edgecolor="black", color="seagreen")
axes[1].set_title(f"log_fare (skew = {df_clean['log_fare'].skew():.2f})")
axes[1].set_xlabel("log_fare")

plt.tight_layout()
plt.show()

---

## 7. ¿Cuánto ayudaron estas features nuevas?

Comparamos tres modelos:

- **Modelo A (NB08)** → `age`, `pclass`, `sibsp`, `parch`.
- **Modelo B (NB09)** → A + `sex_male`, `embarked_Q`, `embarked_S`.
- **Modelo C (hoy)** → B + `family_size`, `is_alone`, `fare_per_person`.

Si las nuevas features aportan información útil, el R² del **modelo C** será mayor que el del **modelo B**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# One-hot encoding (as in NB09)
df_encoded = pd.get_dummies(
    df_clean,
    columns=["sex", "embarked"],
    drop_first=True,
    dtype=int,
)

feature_sets = {
    "A (NB08, 4 feats)": ["age", "pclass", "sibsp", "parch"],
    "B (NB09, 7 feats)": ["age", "pclass", "sibsp", "parch",
                          "sex_male", "embarked_Q", "embarked_S"],
    "C (NB10, 10 feats)": ["age", "pclass", "sibsp", "parch",
                           "sex_male", "embarked_Q", "embarked_S",
                           "family_size", "is_alone", "fare_per_person"],
}

results = {}
for name, cols in feature_sets.items():
    X = df_encoded[cols]
    y = df_encoded["fare"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(X_tr, y_tr)
    results[name] = r2_score(y_te, model.predict(X_te))

print("R² por modelo:")
for name, r2 in results.items():
    print(f"  {name:25s} {r2:.4f}")

print(f"\nMejora B → C: {(results['C (NB10, 10 feats)'] - results['B (NB09, 7 feats)'])*100:+.1f} pp")

> 💡 Importante: `fare_per_person` **deriva** de `fare`, así que para predecir `fare` da una pista muy directa. Esto se llama **data leakage** y aquí lo hacemos intencionalmente para demostrar el poder del feature engineering. En un problema real (por ejemplo predecir `survived`) no harías esto si la feature deriva del target.

---

## 8. Resumen — ¿qué aprendiste?

Las **mejores features** rara vez vienen tal cual en los datos crudos. Las creas tú combinando, transformando y discretizando.

### Conceptos clave

| Concepto | Idea |
|---|---|
| **Combinación** | `family_size = sibsp + parch + 1` — sumar columnas relacionadas |
| **Indicador binario** | `is_alone = (family_size == 1)` — capturar una condición concreta |
| **Discretización** | `pd.cut(age, bins=...)` — agrupar valores continuos |
| **Extracción de strings** | `.str.extract(regex)` — sacar señal de texto |
| **Transformación log** | `np.log1p(fare)` — suavizar distribuciones muy asimétricas |
| **Data leakage** | Ojo con features derivadas del target — pueden inflar artificialmente el R² |

### Reglas prácticas

1. Antes de cambiar de modelo, prueba **agregar features**. Suele aportar más mejora.
2. Si una variable continua tiene una relación **no lineal** con el target, prueba a **discretizarla**.
3. Los strings esconden mucha información — aprende `.str` y regex básico.
4. Una variable muy asimétrica (skew > 1 en valor absoluto) suele beneficiarse de **`log1p`** o de `sqrt`.
5. **Cuidado con el data leakage**: no uses como feature algo que sólo conocerías "después" del evento que predices.

### Lo que viene en el NB11

Las features que acabas de crear viven en **escalas muy distintas** (`age` ∈ [0, 80], `fare` ∈ [0, 500]). Algunos modelos (KNN, redes neuronales, regresión regularizada) **necesitan** que estén en la misma escala. Aprenderás a hacerlo con `StandardScaler`.